# SPS Self-Specialization — Colab Prototype

This notebook runs the latest `main` branch and demonstrates:

`IntegerMultiplication (S0) → replicate → Ollama/Qwen → verify → FloatMultiplication (S1) → reuse`

**Important:** Ollama is started from `/content`, never from the repository directory. This prevents the Colab `llama-server process has terminated: cannot get current path` failure when the repository is removed/recloned.

## 1. Clean Colab environment and get the latest repository

In [ ]:
%cd /content
!pkill -9 ollama || true
!pkill -9 llama-server || true
!rm -rf /content/self-specialization
!git clone -q --branch main --single-branch https://github.com/muhammadnaumantahir/self-specialization.git
%cd /content/self-specialization
!git rev-parse HEAD
!pip -q install -r requirements.txt pytest

## 2. Start Ollama safely from `/content`

The working directory is deliberately kept outside `/content/self-specialization` so recloning the repository cannot invalidate Ollama's current directory.

In [ ]:
%cd /content
!mkdir -p /content/ollama-work
!nohup ollama serve > /tmp/ollama.log 2>&1 &
!sleep 8
!ollama --version
!curl -sf http://127.0.0.1:11434/api/tags || (cat /tmp/ollama.log; exit 1)

## 3. Pull the local coding model

In [ ]:
!ollama pull qwen2.5-coder:7b
!ollama list

## 4. Run deterministic tests first

These tests do not require Ollama.

In [ ]:
%cd /content/self-specialization
!PYTHONPATH=. pytest -q

## 5. Run the real self-specialization experiment

In [ ]:
%cd /content/self-specialization
import os
os.environ['OLLAMA_MODEL'] = 'qwen2.5-coder:7b'
!PYTHONPATH=. python experiments/self_specialization_demo.py

## 6. Where is the new capability?

Look in the experiment output under **[LINEAGE]**. The important entries are:

- `IntegerMultiplication [S0]` — original statically programmed capability
- `IntegerMultiplication-child [SPECIALIZING]` — replicated child
- `FloatMultiplication [S1]` — newly generated, verified and activated capability

Then check **[EVENTS]**:

- `REPLICATE`
- `SPECIALIZE`
- `GENERATED: FloatMultiplication`
- `VERIFY_PASS`
- `ACTIVATE: target=FloatMultiplication`

This proves the capability was created during runtime and integrated into the registry. The current minimal prototype keeps the generated capability in the runtime registry; it does **not** persist it as a separate `.py` file.

## Expected result

```text
Result capability: FloatMultiplication
Result state: S1
2.5 * 4.0 = 10.0

[LINEAGE]
  IntegerMultiplication [S0] ...
  IntegerMultiplication-child [SPECIALIZING] ... parent=<S0 id>
  FloatMultiplication [S1] ... parent=<child id>

[EVENTS]
  REPLICATE ...
  SPECIALIZE ...
  GENERATED: FloatMultiplication
  VERIFY_PASS: PASS
  ACTIVATE: target=FloatMultiplication

USER INPUT — FLOAT AGAIN (INTEGRATED S1)
Result: 15.0
The integrated S1 capability is reused; Ollama is not called again.

SUCCESS: State 0 reproduced, specialized into State 1, integrated, and reused.
```

## If Ollama fails

Run this diagnostic cell and inspect `/tmp/ollama.log`.

In [ ]:
%cd /content
!pwd
!ps aux | grep -E 'ollama|llama-server' | grep -v grep || true
!cat /tmp/ollama.log